<a href="https://colab.research.google.com/github/Hyon2-park/test/blob/main/%EA%B0%95%EC%88%98_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EC%A7%80%EC%97%AD_%EB%B3%91%ED%95%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import pandas as pd

rain=pd.read_csv('/content/종관기상관측_관측지점정보.csv',encoding='cp949')
rain=rain.iloc[:,[0,4]]

In [17]:
rain

,지점,지점주소
0,90,강원특별자치도 고성군 토성면 봉포리
1,93,강원특별자치도 춘천시 신북읍 산천리
2,95,강원특별자치도 철원군 갈말읍 군탄리
3,98,경기도 동두천시 생연동
4,99,경기도 파주시 문산읍 운천리
...,...,...
141,288,경상남도 밀양시 내이동
142,289,경상남도 산청군 산청읍 지리
143,294,경상남도 거제시 장평동
144,295,경상남도 남해군 이동면 다정리


In [24]:
rain['지점주소']=rain['지점주소'].dropna().apply(lambda x: x.split(' ')[:2])

In [32]:
for i in rain['지점주소']:
  print(i)

강원특별자치도 고성군 토성면 봉포리
강원특별자치도 춘천시 신북읍 산천리
강원특별자치도 철원군 갈말읍 군탄리
경기도 동두천시 생연동
경기도 파주시 문산읍 운천리
경기도 파주시 문산읍 운천리
(산지)강원특별자치도 평창군 대관령면 횡계리
(산지)강원도 평창군 대관령면 횡계리
강원특별자치도 춘천시 우두동
인천광역시 옹진군 백령면 진촌리
인천광역시 옹진군 백령면 연화리
강원특별자치도 강릉시 사천면 방동리
강원특별자치도 강릉시 용강동
강원특별자치도 동해시 용정동
서울특별시 종로구 송월동
서울특별시 종로구 송월동
인천광역시 중구 전동
인천광역시 중구 전동
강원특별자치도 원주시 명륜동
경상북도 울릉군 울릉읍 도동리
서울특별시
경기도 수원시권선구 고색동
경기도 수원시권선구 서둔동
강원특별자치도 영월군 영월읍 하송리
충청북도 충주시 안림동
충청남도 서산시 수석동
경상북도 울진군 울진읍 연지리
경상북도 울진군 울진읍 연지리
충청북도 청주시흥덕구 복대동
대전광역시 유성구 구성동
충청북도 영동군 추풍령면 관리
경상북도 안동시 운안동
경상북도 안동시 운안동
경상북도 상주시 낙양동
경상북도 포항시남구 송도동
전북특별자치도 군산시 금동
전라북도 군산시 금동
대구광역시 동구 효목동
대구광역시 동구 효목동
전북특별자치도 전주시 덕진구 덕진동2가
전라북도 전주시완산구 남노송동
울산광역시 중구 약사동
울산광역시 중구 서동
울산광역시 중구 북정동
경상남도 창원시 마산합포구 가포동
경상남도 창원시 마산합포구 가포동
경상남도 창원시 마산합포구 월포동
광주광역시 북구 운암동
부산광역시 중구 대청동1가
경상남도 통영시 정량동
전라남도
전라남도 목포시 연산동
전라남도 목포시 연산동
전라남도 여수시 중앙동
전라남도 신안군 흑산면 예리
전라남도 완도군 군외면 불목리
전북특별자치도 고창군 대산면 매산리
전라남도 순천시 승주읍 평중리
전라남도 진도군 의신면 사천리
대구광역시 동구 효목동
충청남도 홍성군 광천읍 월림리
충청남도 홍성군 광천읍 월림리
충청북도 청주시흥덕구 강내면 학천리
nan
제주특별자치도 제주시 건

In [57]:
import glob
import pandas as pd

folder = "/content/drive/MyDrive/dailyrain_data"
files = sorted(glob.glob(f"{folder}/*.csv"))
print("파일 개수:", len(files))

dfs = []

def read_csv_auto(fp):
    # utf-8-sig 먼저: BOM 자동 제거에 강함
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            df = pd.read_csv(fp, encoding=enc)
            return df, enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"모든 인코딩 실패: {fp}")

def clean_columns(df):
    # 컬럼명 공백 제거 + BOM 제거 + 탭 제거
    df.columns = (
        df.columns.astype(str)
        .str.replace("\ufeff", "", regex=False)  # BOM 제거
        .str.replace("\u200b", "", regex=False)  # zero-width space 제거(혹시)
        .str.strip()
    )
    return df

def find_stnid_col(df):
    # 1) 정확히 stnId 있으면 그걸
    if "stnId" in df.columns:
        return "stnId"

    # 2) 대소문자/공백/특수문자 제거해서 유사 컬럼 찾기
    normalized = {c: "".join(str(c).lower().split()) for c in df.columns}
    for c, nc in normalized.items():
        if nc == "stnid":
            return c

    # 3) 그래도 없으면 후보 탐색 (stn 포함)
    candidates = [c for c in df.columns if "stn" in str(c).lower()]
    return candidates[0] if candidates else None

for fp in files:
    df, used_enc = read_csv_auto(fp)
    df = clean_columns(df)

    stn_col = find_stnid_col(df)
    if stn_col is None:
        print("⚠️ stnId 계열 컬럼 못 찾음:", fp)
        print("컬럼:", df.columns.tolist())
        continue

    stn = int(pd.to_numeric(df[stn_col].iloc[0], errors="coerce"))

    region = rain.loc[rain["지점"] == stn, "지역명"]
    region_name = region.values[0] if len(region) > 0 else None

    df["지역명"] = region_name
    dfs.append(df)

rain_all = pd.concat(dfs, ignore_index=True)

# 확인
print(rain_all[["지역명"]].head())
print(rain_all.columns.tolist())


파일 개수: 93
⚠️ stnId 계열 컬럼 못 찾음: /content/drive/MyDrive/dailyrain_data/종관기상관측_관측지점정보.csv
컬럼: ['지점', '시작일', '종료일', '지점명', '지점주소', '관리관서', '위도', '경도', '노장해발고도(m)', '기압계(관측장비지상높이(m))', '기온계(관측장비지상높이(m))', '풍속계(관측장비지상높이(m))', '강우계(관측장비지상높이(m))']
           지역명
0  강원특별자치도 평창군
1  강원특별자치도 평창군
2  강원특별자치도 평창군
3  강원특별자치도 평창군
4  강원특별자치도 평창군
['stnId', 'tm', 'sumRnDur', 'mi10MaxRn', 'mi10MaxRnHrmt', 'hr1MaxRn', 'hr1MaxRnHrmt', 'sumRn', '지역명']


In [59]:
rain_all.to_csv('rain_all.csv',index=False)